# MLflow Example: Tracking Two Model Training Runs

Dependencies are installed in the project's `.venv` (`mlflow`, `scikit-learn`, `pandas`, `jupyter`, `dagshub`). Select that venv as the notebook kernel.

## 1. Baseline: training without MLflow

In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [2]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

def train_model(n_estimators, max_depth):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [3]:
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389

Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


## 2. Now adding MLflow

### Tracking URI
**Local (steps 2–3):** start the UI first from this folder with `mlflow ui`, then the notebook logs to `http://127.0.0.1:5000`.

**Remote (step 4):** comment out the local cell and run the DagsHub cell instead.

In [4]:
import os
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

# Local tracking server (started with `mlflow ui` from this folder)
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://127.0.0.1:5000


In [5]:
# --- Step 4: remote tracking on DagsHub ---
# Get this code from your DagsHub repo: Remote > Experiments > "Using MLflow tracking".
# Uncomment, fill in your names, and run it INSTEAD of the local cell above.
#
# import dagshub
# dagshub.init(repo_owner='YOUR_REPO_OWNER_NAME', repo_name='YOUR_REPO_NAME', mlflow=True)

### Create the experiment

In [6]:
# Created if it doesn't exist, and set as the active experiment
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)

2026/11/16 17:24:25 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_V1.0.1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794846265873, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794846265873, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

### Log parameters, metrics and the model

In [7]:
MODEL_NAME = "random_forest_model"  # artifact name of the model inside each run

def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        # input_example: lets MLflow infer the model signature (needed for clean serving later)
        # skops_trusted_types: MLflow 3.18 saves sklearn models with skops (safer than pickle) and
        # refuses unknown types. We trained this model ourselves, so we trust the tree structure.
        mlflow.sklearn.log_model(
            model,
            name=MODEL_NAME,
            input_example=X_train[:5],
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"Logged run '{run_name}' (id={run.info.run_id}) "
              f"n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")
        return run.info.run_id, accuracy

In [8]:
print("Training and Logging Model 1...")
run_id_1, acc_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...
Logged run 'random_forest_model_run_1' (id=caf103246a7d45618c733c393ba68721) n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: http://127.0.0.1:5000/#/experiments/1/runs/caf103246a7d45618c733c393ba68721
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [9]:
print("Training and Logging Model 2...")
run_id_2, acc_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")

Training and Logging Model 2...
Logged run 'random_forest_model_run_V1.0.2' (id=7033da8ca62d49e88be662da96d6c1c8) n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: http://127.0.0.1:5000/#/experiments/1/runs/7033da8ca62d49e88be662da96d6c1c8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## 3. Register the best model in the MLflow Model Registry

Each call to `register_model` with the same name creates a new **version**. MLflow 3 uses **aliases** (e.g. `champion`) instead of the old Staging/Production stages.

> A warning *"Run ... has no artifacts at artifact path ..."* is expected in MLflow 3: models are stored as *logged models* (`models:/m-...`) and the `runs:/` URI is resolved to them automatically.

In [10]:
REGISTERED_MODEL = "RandomForest_Digits"

best_run_id = run_id_1 if acc_1 >= acc_2 else run_id_2
print("Best run:", best_run_id)

mv = mlflow.register_model(f"runs:/{best_run_id}/{MODEL_NAME}", REGISTERED_MODEL)
print(f"Registered '{mv.name}' version {mv.version}")

client = MlflowClient()
client.set_registered_model_alias(REGISTERED_MODEL, "champion", mv.version)
print(f"Alias 'champion' -> version {mv.version}")

Best run: 7033da8ca62d49e88be662da96d6c1c8


Successfully registered model 'RandomForest_Digits'.
2026/11/16 17:24:46 WARNING mlflow.tracking._model_registry.fluent: Run with id 7033da8ca62d49e88be662da96d6c1c8 has no artifacts at artifact path 'random_forest_model', registering model based on models:/m-de760952a4b9489ab8e7309f2d55f987 instead
2026/11/16 17:24:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Digits, version 1
Created version '1' of model 'RandomForest_Digits'.


Registered 'RandomForest_Digits' version 1
Alias 'champion' -> version 1


## 4. Load models back from MLflow

In [11]:
# a) From the registry, by alias (what a production service would use)
champion = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@champion")
print("champion predictions:", champion.predict(X_test[:10]))
print("true labels:         ", y_test[:10])

# b) From the registry, by version number
by_version = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")

# c) Directly from a run (same URI format as `mlflow models serve` in step 5)
from_run = mlflow.sklearn.load_model(f"runs:/{best_run_id}/{MODEL_NAME}")
print("accuracy of reloaded model:", accuracy_score(y_test, from_run.predict(X_test)))

champion predictions: [6 9 3 7 2 1 5 2 5 2]
true labels:          [6 9 3 7 2 1 5 2 5 2]
accuracy of reloaded model: 0.9722222222222222


## 5. View the results / serve the model

In [12]:
print("View results:   mlflow ui   ->  http://127.0.0.1:5000")
print("Serve the model (step 5):")
print(f"  mlflow models serve -m runs:/{best_run_id}/{MODEL_NAME} -p 5003")
print(f"  mlflow models serve -m 'models:/{REGISTERED_MODEL}@champion' -p 5003")

View results:   mlflow ui   ->  http://127.0.0.1:5000
Serve the model (step 5):
  mlflow models serve -m runs:/7033da8ca62d49e88be662da96d6c1c8/random_forest_model -p 5003
  mlflow models serve -m 'models:/RandomForest_Digits@champion' -p 5003
